In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
#STEP 1: GENERATE DATASET (600 rows = 200 per class across 3 sensor features)

np.random.seed(42)
specs = {
    "Lithium_Battery_Device": [(0.72, 0.98), (0.30, 0.65), (0.52, 0.95)],
    "Circuit_Board":          [(0.42, 0.74), (0.58, 0.95), (0.10, 0.38)],
    "Plastic_Casing":         [(0.05, 0.38), (0.02, 0.28), (0.05, 0.28)],
}

rows = []
for label, ranges in specs.items():
  for _ in range(200):
    vals = [np.clip(np.random.uniform(*r) + np.random.normal(0, 0.03), 0, 1) for r in ranges]
    rows.append(vals + [label])

df = pd.DataFrame(rows, columns=["xray_density", "metallic_ratio", "thermal_signature", "label"])
df.to_csv("ewaste_sensor_dataset.csv", index=False)
print("STEP 1: DATASET GENERATED")
print(f"Total Rows: {len(df)} | Class Balance:\n{df['label'].value_counts().to_string()}\n")
print("Sample Sensor Readings:\n", df.head(3).to_string(index=False), "\n")

STEP 1: DATASET GENERATED
Total Rows: 600 | Class Balance:
label
Lithium_Battery_Device    200
Circuit_Board             200
Plastic_Casing            200

Sample Sensor Readings:
  xray_density  metallic_ratio  thermal_signature                  label
     0.784024        0.364165           0.553347 Lithium_Battery_Device
     0.934414        0.289778           0.583109 Lithium_Battery_Device
     0.720727        0.430441           0.770255 Lithium_Battery_Device 



In [13]:
#STEP 2: STRATIFIED TRAIN / TEST SPLIT (80% Train = 480, 20% Test = 120)

X = df[["xray_density", "metallic_ratio", "thermal_signature"]]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print("STEP 2: STRATIFIED DATA SPLIT")
print(f"Training Set: {len(X_train)} rows (80%) | Testing set: {len(X_test)} rows (20%)\n")

STEP 2: STRATIFIED DATA SPLIT
Training Set: 480 rows (80%) | Testing set: 120 rows (20%)



In [14]:
#STEP 3: TRAIN REGULARIZED DECISION TREE (prevents overfitting & underfitting)

model = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42)
model.fit(X_train, y_train)
print("STEP 3: MODEL TRAINED")
print(f"Decision Tree Fitted | Depth: {model.get_depth()} | Leaves: {model.get_n_leaves()}\n")

STEP 3: MODEL TRAINED
Decision Tree Fitted | Depth: 2 | Leaves: 3



In [15]:
#STEP 4: EVALUATE GENERATLIZATION (Compare Train vs. Test vs. Cross-Val)
train_acc = accuracy_score(y_train, model.predict(X_train))
y_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
cv_acc = cross_val_score(model, X_train, y_train, cv=5).mean()

print("STEP 4: MODEL EVALUATION & SAFETY RULES")
print(f"Train Accuracy: {train_acc:.1%} | 5-fold CV: {cv_acc:.1%} | Unseen Test Accuracy: {test_acc:.1%}\n")
print("Classification Report (Unseen Test Set):\n", classification_report(y_test, y_pred))
print("Learned Safety Rules:\n" + export_text(model, feature_names=list(X.columns)))

STEP 4: MODEL EVALUATION & SAFETY RULES
Train Accuracy: 100.0% | 5-fold CV: 100.0% | Unseen Test Accuracy: 100.0%

Classification Report (Unseen Test Set):
                         precision    recall  f1-score   support

         Circuit_Board       1.00      1.00      1.00        40
Lithium_Battery_Device       1.00      1.00      1.00        40
        Plastic_Casing       1.00      1.00      1.00        40

              accuracy                           1.00       120
             macro avg       1.00      1.00      1.00       120
          weighted avg       1.00      1.00      1.00       120

Learned Safety Rules:
|--- thermal_signature <= 0.46
|   |--- metallic_ratio <= 0.43
|   |   |--- class: Plastic_Casing
|   |--- metallic_ratio >  0.43
|   |   |--- class: Circuit_Board
|--- thermal_signature >  0.46
|   |--- class: Lithium_Battery_Device



In [17]:
#STEP 5: LIVE ROBOTIC ACTUATOR INFERENCE

live_items = pd.DataFrame([
    {"item": "Old Smartphone", "xray_density": 0.89, "metallic_ratio": 0.48, "thermal_signature": 0.78},
    {"item": "Router PCB",     "xray_density": 0.58, "metallic_ratio": 0.82, "thermal_signature": 0.18},
    {"item": "Keyboard Shell", "xray_density": 0.14, "metallic_ratio": 0.08, "thermal_signature": 0.11},
])
commands = {
    "Lithium_Battery_Device": "PICK -> Fire-Safe Discharge Bin",
    "Circuit_Board":          "DIVERT -> Precious Metal Bay",
    "Plastic_Casing":         "PASS -> Plastic Shredder",
}
live_items["prediction"] = model.predict(live_items[X.columns])
live_items["robot_action"] = live_items["prediction"].map(commands)

live_items.to_csv("live_conveyor_predictions.csv", index=False)

print("STEP 5: LIVE ROBOTIC CONVEYER INFERENCE")
print(live_items[["item", "prediction", "robot_action"]].to_string(index=False))

STEP 5: LIVE ROBOTIC CONVEYER INFERENCE
          item             prediction                    robot_action
Old Smartphone Lithium_Battery_Device PICK -> Fire-Safe Discharge Bin
    Router PCB          Circuit_Board    DIVERT -> Precious Metal Bay
Keyboard Shell         Plastic_Casing        PASS -> Plastic Shredder
